# 🏏 Naive RAG — Cricket Q&A with Groq + Gradio
> End-to-end RAG pipeline: CSV → Embeddings → FAISS → Groq LLM → Gradio UI

In [1]:
# =========================
# STEP 0: API KEY SETUP
# =========================
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')


In [2]:
# =========================
# STEP 1: INSTALL DEPENDENCIES
# =========================
!pip install -q -U langchain langchain-community sentence-transformers \
    langchain-text-splitters faiss-cpu langchain-groq gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.4/512.4 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests

In [3]:
# =========================
# STEP 2: IMPORTS
# =========================
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from IPython.display import display, Markdown
import gradio as gr


In [4]:
# =========================
# STEP 3: EMBEDDINGS MODEL
# =========================
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Embeddings model loaded")


/tmp/ipykernel_12132/4294909453.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embeddings model loaded


In [5]:
# =========================
# STEP 4: LOAD & SPLIT CSV
# =========================
loader = CSVLoader("/content/cricket_stats.csv")
documents = loader.load()
print(f"📄 Loaded {len(documents)} documents")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
documents = text_splitter.split_documents(documents)
print(f"✂️  Split into {len(documents)} chunks")


📄 Loaded 41 documents
✂️  Split into 41 chunks


In [6]:
# =========================
# STEP 5: FAISS VECTOR STORE
# =========================
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✅ FAISS vector store ready")


✅ FAISS vector store ready


In [7]:
# =========================
# STEP 6: LLM + PROMPT + RAG CHAIN
# =========================

# LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

# Format retrieved docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Prompt template
template = """
You are a helpful cricket expert assistant.
Answer the question using ONLY the provided context.
If the answer is not in the context, say 'I don't have enough data to answer that.'

Question: {input}
Context: {context}

Answer:
"""
prompt = ChatPromptTemplate.from_template(template)

# RAG pipeline
rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)
print("✅ RAG chain ready")


✅ RAG chain ready


In [8]:
# =========================
# STEP 7: TEST QUERY (Notebook)
# =========================
query = "Can you give any 3 Indian cricketers?"
display(Markdown(f"### 🏏 Query: {query}"))

response = rag_chain.invoke(query)
display(Markdown(response))


### 🏏 Query: Can you give any 3 Indian cricketers?

Here are 3 Indian cricketers:

1. Jasprit Bumrah
2. (I don't have enough data to answer that. - I need another player's data to provide the third one)
 
However, I can provide the third one if you provide the data for another Indian cricketer.

In [ ]:
# =========================
# STEP 8: GRADIO UI
# =========================

def ask_cricket_rag(question, history):
    """RAG function for Gradio ChatInterface."""
    if not question.strip():
        return "Please enter a valid question."
    response = rag_chain.invoke(question)
    return response

# Build Gradio ChatInterface
with gr.Blocks(theme=gr.themes.Soft(), title="🏏 Cricket RAG Chatbot") as demo:

    gr.Markdown("""
    # 🏏 Cricket RAG Chatbot
    **Powered by:** LangChain · FAISS · Groq (LLaMA 3.1) · Gradio
    Ask anything about the cricketers in the dataset!
    """)

    chatbot = gr.ChatInterface(
        fn=ask_cricket_rag,
        chatbot=gr.Chatbot(height=400),
        textbox=gr.Textbox(
            placeholder="e.g. Who has the highest ODI average?",
            container=False,
            scale=7
        ),
        examples=[
            "Can you give any 3 Indian cricketers?",
            "Who has the most Test wickets?",
            "Tell me about Virat Kohli's batting stats",
            "Which player has the highest T20I strike rate?",
            "Who are the all-rounders in the dataset?",
            "Compare MS Dhoni and Kumar Sangakkara as wicket-keepers"
        ]
    )

demo.launch(debug=True, share=True)


/tmp/ipykernel_12132/3697398728.py:13: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="🏏 Cricket RAG Chatbot") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2faf42aecb4edb908a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
